# Regenerate s1K reasoning and answers with DeepSeek Pro

Run on **Colab CPU**. Add `HF_TOKEN` (read access) and `DEEPSEEK_API_KEY` to Colab Secrets
and enable notebook access. This notebook contains its helper code: no Git push or GPU
is required. Use a fresh runtime when opening a new version of this notebook.

Source: `Seungjun/dp_removed_s1K-1.1`, pinned to the same **996 rows** used by the project
before the SFT length filter. Every original column and row is preserved in the full export.

DeepSeek Pro solves the original **question only**; it is not given the old reasoning,
answer, reference solution, or grade. Thinking is enabled. Its native `reasoning_content`
becomes `deepseek-v4-pro_reasoning`; its final `content` becomes `deepseek-v4-pro_answer`.
The final response must contain **four named sections: Planning, Evaluation, Reflection,
and Exploration**, followed by the final answer. Evaluation contains the full worked
derivation; the other sections explain the method and problem-specific checks. No length
target is requested, and mistakes or alternatives must not be fabricated.
There is **no answer-correctness filter**.

**The intended new training target is `deepseek-v4-pro_answer`: the complete structured
worked solution.** `deepseek-v4-pro_reasoning` retains raw API thinking for inspection.
This notebook does not change the existing SFT loader to use either new column.

**Updated format:** reopen this notebook in a fresh Colab runtime and rerun the smoke test.
The new prompt/code creates a fresh output folder, leaving previous results intact.
Seed 42 selects the same 20 questions, so you can compare the old and updated smoke runs.

1. Run setup, settings, and source-loading cells.
2. Run **Smoke test** for 20 reproducibly random examples and download the five-column CSV.
3. Use the separate **Full run** cell to generate the entire dataset, reusing completed rows.

A default **Run all** runs the paid smoke test only; the full-run checkbox starts off.
Both modes save each completed response immediately to Google Drive. Rerun a cell after
interruption to resume; unsuccessful rows are attempted again, completed rows are reused.
Use only **one notebook/runtime at a time** for the same output folder.

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/LG-AIME-S1-DeepSeek"

In [ ]:
# @title Install CPU dependencies, mount Drive, and read secrets
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "httpx==0.28.1",
    "PyYAML==6.0.3", "tqdm==4.67.1"])
from google.colab import drive, userdata, files
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not HF_TOKEN or not DEEPSEEK_API_KEY:
    raise ValueError("Enable notebook access to HF_TOKEN and DEEPSEEK_API_KEY in Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-s1-regeneration")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVzis0wOThMAACNAAAAZAAAAcGlwZWxpbmUvcmVnZW5lcmF0ZV9zMS5web07a3PbRpLf9Stmx1e1oAzBcpLd2tCHc3kT5daVZDdle+/qimZQEDikZgUCCAbQIyr99+vueWAGAEUlV3X6YJPz6Onpd/c0Oeff1FXRt62oOvatEM1HIa5ZK3JVV7LasZ2oRJt3sq7esKpmeaVuRXtW1LCh6CqhFNvKshMtrE045ycnct/UbcdydV8VsrZfi7q5d5/Vjf3Y5tWm3rtvwn7q5F6cbNt6zzZ5J/AbMzP2e0xrfq0rs07sc1kmfSdLZZc2easErs+6OrP79Oom765KeWlX/gRfHeZXXdfcneh1Rb3f15VdBtfxRhNZ24m8aUS1yf4FNCtjtpE7oboYiejGblsJeOAX/3N5cnKyEVu6iuryfRMtlicM/lrR9W3lLptU9W1k7wuXLBaJVPW2bvd5Fy0MkJu8lHRbVfdtIaK2vlUx4Fpt5c6AVY0oWGrGVlwv5Guak1tWiop2LdgfUlq74uIO/uvEJsNxvmZ1a643XggwOxChTM/ytT6Q7pJLJdh/5WUvLtq2biP+kc4FQNutaEGAkKbdlWCNrCqxYRuBwPK9rOA2G6Zef4+EyJXo+IKgwsUZHM9khf+p4SicqHIUlspdsqjLfl8B7skNoqAsie0f3LuqgbdKVsCCShMu2YkuQkiLmKmuXeC9cRVMrXB4ncCobMawZq+75T9KpVCXNL1ZJ+66JXtAOI/ccq+sQVgM5zTqIOH1tajMEVYZkAxOwmmTGTw5ymGzEKb9fZHmXyuamq9jIl/qWApAcFA1pezMKH2GwcnFfRqIG6BnXaUWtv6KoOhOqb6ZFnXgIKIkQaYMSnriSXn2tQQnDBVhpL3P8i0YpEyhGG1URGw3VDTMpiFPQjWc8+SchgDEZG6f30UwH7Mt0K4zMDUW4q4QTedxfNgcQJqDNmuiDPTEswrsjGwEDUXmXO/saDg8Zp/uG/vxHzeiBYRv6etiFhW89MlJUeZgyb8D8pfvfnqv5fZDX+Fx/l4w8O960FSYKMgnxOxSliXKNiiIrIhlAPkX0LOOCdyp2L6Hz6qrG7gtzN7W7bX2FMQx7VAybcsyUBRUUT0YEyTZgvUBXuJxasDjmytRXDOUq8tSMDOPutoXcC/BwLjex8Rs31eh4XCu7Zce0O3uCZl5S2Dx8E2AHpvqv6Hniu+1tmdbsF9l1grVAOJWB6/AJ8Ckk/hWJLBuAw6hjVr+84sXK/a5W7+MVj9/bj9X65dvFzRw+rl9+288ZgNhkh//+cOn9z+8//uFEQdUXAS72ie7tu6b6PXCIkl2cY9G0R5vTD5RG/c4F6ChgFWfkH64qd6WaKcX8cnKzFJg04OtKFC66zar+y6rt/BpI1rPjsvYIoX4iarfY7ghIoupR2E4DXC1EyvJXrLXaIjzFpwgIk8j7N/Jj7n9TJRgj3FIE29Qnst6cw8A9fDKbEjwTosl/OtsvNvwgr1D+1OUPeGrN7LLeyY7Jcot29RAO5SRpm76Em5BXu3irilrHUJZOU3GOAA/ya6CDERv5X7x82d1Gr1dfj79fLp4+x0KkjltCUKAexar8yl+RoBxfuTjQo6JfdPdW3YtOVCN2L6S68Cu6l3OOnwQQywItkGfgCqcZbKSXZZFSISY+XY6ZsD2Bjjf1nXnsRJXJsbyY2iYbCDyxA86rAjXaVCTlb4nwL+xv3BnxD4YH7YWVwDsza+4wVhbJL5eTQXc6LIheTQyGnZVTBpOdgOlz5kw1K6vJt4TVuXV/QRWaHtUoNEKNcaBnYOoz+2Gs/XhY08wCdAuTNQHx/QY8KlOVkXHyKKdWYvmjG4nu1IoHsgh3oW/eMFQuGhem9YqJHUDQU0DNKbr6GX+lY6h+d1T+Gi/c4kQUWHhMgCaokx96IBu3sixDMBQyGQYAMteb0QJ2JqrPHCURQX50tnNV2cAFnRzGNqC0lzxx2N3+Gel+gaDOUDQpV/6oMUEAee7MrEFknU+LuDm8fwrubvC/yHEeMbh8D1nAwLdlayu0bQZ8BPT4vtGRKiwuWNxj8GdrLS8T+fAKr8+ho23wTEvB1uqZCdvkI8dWKD2OFJw9QwDQQlCOUYqmAOkzo8h5W3wkKpqyFR2+dN4gStKh1Mp5KVDU/bl119+8frPVqvPk6//5JZCnJQ1etVRgr03AZeTGzqDFULakExDmxMkCtEyDPDQMZtYWZ97PlBLVplditds4Tx/7XHyvR/FhHAgoDal1bkjgA7h0aPI8LBgHinq5p6PBGydolDUEOmOLYDaAy3R+os7iMDRvq2fkDsCMYia474GDR9cav1cXOl8puhspuSvgo+84mALdIbL0tBIvWQ8c2t4uNcE3Yc36gV8bNHnD9ZpON57BriZHKXsR8nwnzrUAJtk8neWl1hPuYeUB9zRG+DltqekuqtZDWkOFVVshr2VotyoMcWIpBlG2wVFvbr4lHyg/6J57oMp3/A1RNLEiGiEdbUTkePsItbs9uPFf0AWwkxpBLIRAYhC8l1AOIcVKPDul6KsMR0wvgl8Wb/PMaORG8yyIDsZeIAY7GFUh5KpLcSssHoVZZBvlCLLFgmVnbDGEAFGNCfrZHZ67amhOW4A+2Dz/6WvGOBZQjxgOhx4HMspRH4AlRDxg0H2ivG2B4sIHwL5e+WQWS1f/3k9hZbsrzeyjSB3hmUq/dT2ItZikdXX9HVAYZ9XcouGJ2UPAfO4PQQvYD7C5X7fncNiCA8EzcIKBkcbKnGb2TrVkq1m1Sye0a5RFYajja4g1Mk60CjaC8nRTnRmuUUlgDEC0ea3GbAjs6HAaOsEq3C3l2tnui4MG7/LIQMbLcQjULcyF3xMFz66T0PFNBpk6hUGOZq7CU5B1GO/j0TwX2AVMIVKWbA77zrMhBTtLvnYvMJdNmQmXBk38qFhEt7ksgUSy3IkdbQOE0AteMNFyAwSZBcLm4NCg+gBWOkFK2DuRtzx9ZpQwqHwuAyUicoaqabjicvPVF8UQGqxMQkaAZpWLrwzqfxJy2LAfkFfObi6rgejytIUGY0K0AnuHZPfCJsCEn7eEX6BfEJFWjtLfZuxzq45Sh5aTu8QOk29zSFLVWXdEQY+drTmVnZXlpSYRoKdvg65kiME+7KRqBLCrsiV8vTOCgyrDZls0W5fQzwASlNEi8ViyuZwTzrZAw55mpusJunekWBtQg2z1HCsKKXQEUwoG0dzI1PAGBlXbcuXgWEHM2qNCsw88O6+EfCBiwod3oY/ji3RONlZHkqCEDIFussghh7B84Lw5TQsBxiQW4t8f8Bg7cGk5Tsy51PaP/C2Luky6l6BUeHak6DXD13JkPI+TgvoA5RegeGcwsAwY0UcmuG/k6fJw8eKE6epAD8+dz1nbtFKGeNIARvFOTPp00v2ejGnIVqsB3UbCT0W7MRmRtJH9WkykyBaRrWXzFgkm72A95iVswMvE1xDhF05UtSrrsfOF4DiZBCdEUhDALjk4wgxTEDgqBwF3z4aONphDd3ZYH9m8hhABBtMj7UrJieLnk7UZl6eNHKmGJIaXmjFTiCLBhP+qrjKu1fGdlMlK2Zok1MqKR4AqM0rvopmxgVoG6sPSvQYhAMb8SRGCQX5WFYzYKL5A/ENyIdPXn+6VOOV9A2W/KInT87kJkWo2o3JDV/EwxwJjTdtSi/zQqS1Gyv7kPA3LVhZb+d0Eg/q0Wp4q+j7HPzpFYurGmJFjMhhN+gzfUV9Pl9P1hrzhEVSWrayBotP1zoDat8TYJdZrXH0gzxtgOAiwQo3PgEO15fqyuGhl+tBk47ObLLRBGybZ6WFmqKVrhsOiG8oyx7hDTlhXs4cgH+4xX/lHahAJVacvqzrcpiwxdZngQueihys8LFoCmiGfv5DmHudGL2PHUDIYPBbi9m/U7kMldKJOKUWk4Dtqf42r1iGc+k8O+f3BCRJg28HtJfMTgomQNdYLFWwuhGSnZ6Mhhh3XgBgl5NbvUFWbstz9JscCSX7qRVwquSCTPXbrSzQcGfGqoC90k/56KsudcIUuiXzFEzdK8nfPn366SNdl+op4FtwfuoyNEngfJhNnmXSfaTNbkT6q/O/xOyr86/hny++fqRakJ78j5T96fx8KubGTVJThQU5WfWCfVvTkrLeuRIiPtSJFp+YQIwxcQdPG1PJsr2UkPi296ACLVZa6AEaI1QIV5IjQm6EgxJT2geUpv/TLUdqsge94nFGGsKIYK4HISCvuYAVclh9Rqv5jIEAIbM8SpG0817/uQlMwPq53AOzmelEPJeT4NKDwEdUiZ8s7Y6ubXsatCR/guhTYYXAdjfoQEh3JRyUa19KMTX/3czHJAW5t0iyDB9Js2weWb8B43txbz69x4B12pbxrgN/cNl34v/rFvzHvEQLJzYfjBDSq++xy9nYD1jYKO0yTDEepLzuITGfyMWZDe9j9uVMrks1gnE2j39ogtEkHOqnCltTttw9eWjdNJj+0YtS/7h+fANBCHaJBIaCPym09o9f5iX6dXLlVj+U6DrsKYixVgt5kWj7KmHfGJu/0a1MeMVNwifXG5iJIG12kaYz71KHiOB35EQDCTpUEXQWZ9rg2ecqcXeV95gmvXGOyuD4LBrwvBXmNiDI2MnVsbBkZogDVMAqPNWuxczNAxs/K+V9WwXDl3lxXW+3aJBkFR169nnaqrBT9gU7PbUoh1i9YH8Doe1b9oHM7zs0v0zcAFNvkbOQN6Fii41iuEjcgc5oATqzqJlHttCtHCoQmU2xbxPB6o1KMqa9QgxlOioV4y2zawFG9DTGx+sdENpWvTtrHdO/A4ZeTsg5/0D80NU/pYR6Y96/trksrSDUFcsZFtKoe6vWTV0J+ysqt27X8h5lXaOU4WtQdpx70Anl9dOVVAxlRQ6POXmBT6jT50Bz5WOvRN9eXPz08eLi+wwsQ/b9xf8gaBvg+kDdkw/1XJmvwaG2U8LOUaMEDtoB26Ah8dWTut0wtNHbTGlGd2VIjIfsrue+9mFbrW2Wso9Y9v1gVPb06ryBT+hyde31cmk1wtQOJWNYNS5C6ObbXzb7JO8719CMAyehaB8rBHoI2uAD0xejCz/A13EdaC4AGZdTsPyrL0stcnJCYisx2r24SrccBRWaFHitqKvBlaQ+b1HXyHqkoDHRBNKUrQebXzdCFelgmYeGKXDFPZyR2udkjm3iCg1iSg1jRq1DpL2wTgdD73DgG6roTGOvy1yJrG/LlKyi/YZm0gSc6QN1b9at/NU+Y235XwWY+ZY9GI17HJdhSWh0zJU+XZHyrZH7NAVWyr3sVKov9AN9QRuJpYbKpKTpTG/Js7w2wrkG4oNO3YgjEEdMpChMF8sOVenQRmP3KsjhgeobCQrGfCgsRnQPR+B+sdRW48M6/MGtIM426nv99CowahkW/7byLrJRwP9ZzCd/kKqD+F6lVPVcnEwlyBonaw8gJKOuZxiPLE01BtlQcp493kUEo94jX6EXi2nVa7b8in+h097lENu10SlhPFveysvyACTqYoN99JqP+w/zHqeTAiPM8kAd9Am0YhMzZTr7IAEPHyBn0QRLiWZQ6h5VdAtT/FBkirJWYtZYH3xixAip7caBC9ZSw5jk4k7/XqYsmSvim4ZRMnNVDzO6ZqWInBis9CBaFGoq0OyWPKROibGlQY2jEjx06IrTjRwQ9W8B8nO68a6r+rYy9yFYvy2OoNOpTEkH6yqh2faHdKYT4BhGH6kPyOKDDWi90n3FoNFIPOpeMY1CfB4XuvsYFd1+7lpYhihmcbzvErnko9QAh7DkgiE0ll90AGP6fcw3v/FbawB2Joze5Pd+yUmv+C3dHlouxCgMyvsddmrQk5M3HBjreW7YJ6jDT+PBeq+MPbauU3PusLKP3A8TZTw9HT/7TS3ybFfG0qXuQwfYelqwRBNwAGLQIjKAM31hz4I1Io7ljbuu+XFSNn7dM49M9tCg+QCmjVs9VBV2f6NHhxDc6EHiGChdTQlB6LHjWPiF5TEWwVwMwnkUnPemNQLmzzwfjH1AnYVk3sI8Rrb4M0vjAjCImI28p4HEoKD9fo8FWnzVdQ4Ajg+j8aHyHs6wswGB2RtyN01XskuZJ2qmJWu2D8tCOdzFNJTc4da6RR51fcYX9OY5btUG736Y6T086j6gcQsQVYlcI4uZpUPWYwLq3+thnc00MGTS/izFe58ft2VgjW9JfOvdDxxjdk5InRNOPZ5KqIbK6/+w0vYw2NaJgV8QiHiDlGe5/goHbraryziDV8ApfUHb0WW+LmY2lf4uMg+mkSt2tuYp17wcme5y8kMQ10AxISI5GNzit1bETI9AfrUzP/1ytncyZw3pAUV9fgdgmC1DlIhprqMLMiZvJYBJCnUTlv2wLFa3WhtxY4IRWEZPT3cRx/VJt29G5TyK0tzOpAZrHHH84QGE4TWKX8r7bnv2lzMldxw7Xm9LWYmUc8qvttOAk/iJuTme960suv+mgWgb6y5eIndK/6Lbh8RS5ZTTpVzuqrqde53TMBP6T+e+MzG2vwg9bOQc8mKeTMCPpswLESGxvJ9ggllYPsmFiJvftSbACywzGDnVVWFrUPUTol2pV4xo74k+nhGzGZRNM98DxxVgtvRCq1Z8aTXKOdrMLDygTI8n/wtQSwMEFAAAAAgAAAAhXGALJ9wDBgAA0gsAACMAAABjb25maWdzL3JlZ2VuZXJhdGVfczFfZGVlcHNlZWsueWFtbG1W247bNhB9368gNo+1tb47Vp6CNgGCXrZo0ocCBQxaHEmMKVIlKXvdy7/3DCXZ3qJPu6aGczlz5gyD63xB+YMQnlqXi8/U2eprZ59Uu/fUuBOpfZh/P51n82Rz0kE7m4uNmm+3h3Izk/PdWhXLYnFYLRVt5Fsqy/liNVvLebFbLXCpcLbUVS4UlbIzESehNTrmInqpLX7SS0tFRCDvziEXu92mvxXJxr3SFQUYb7br1aJcU7lclDvalsvteiaXhx1C0Xo122zVlortbLaabxaS3qqlLBazxW6zW+xkuVmuHwpnusYGLvWPDi5TGeN/OHReV9pKg7JlcFbblDK1gei4j7W2RxztkfNXJOv85f6KtOFM/s5exkhNGx9kqzngQQbad97koo6xDfnTEz5ko3VWuAZGjVNkbj6mp9W09S6hPiS0B7bOA4xaVzU+RNfu21zMsxl+vBHPLXnJ1UgjCtIGNwTsb/fFN6LPdCJ+ev4iJL4kAEiJKH1FURiyVayz5O798GtqdKPZxlNonQ0kdBClkVVFaiIsncijlSTZRAZ4RTmtoUhorOT/2F0jX/bRHQkNEPPlfLYdmFF03pMtLjjlrjfa7oek9hoE8CegGwiGChdn2SLB0X+PuiHXxdvn+dvZbAjlKXpNOFumC9Ff4E5Hfe9tcf3EN67Hm9lDaJBpnpiZCmBj9oy2qFyA1AjbIjLAbWTMh5y051kBOdCBRDMhpuJxnomfjbQM/+N4tsjEBxTWpWZdT5eZ+IVK0zu4nq5g+9Ia5wdjUALEysXfU1h8duZEItYkQtdiqNABfD8YaoS24oOtjA51Jn7DkIuSmXrrYdOFKA6EdgUy5ZTHDdMIB2fnj6S4WgwMx4RvHSWcJjJFkkXNXJLWIbDvaStqdwYd+c6Q0JBHJr7gB9ihYHrWxghnzSUhBoomW2bTq9xAKhfFhZMO1EpUnszSAD7dyFzUgJVMiuAJuYGYtjCdIh4NRC07hENcfUrYTcSRLqKQpuhMOggT8RUo6FIX429pFfxScQwMYHLyGjYdGa53iKBcShOyaWSRam5QYKzTAHAeLJSQCEa4axrpL8KVd7OIYWX+AHEygc5cQ/YAv8++klb/Sf8XvbMMI1hZRHNhg8BNAU4/Sn9U7mxFDaThHaWk/IGt87iTs+c3b8QdG3HwSUFhdZk8icpJMxEVemJFiQBwgTpCEuk4QOPJ0EnamCqkNoaenHLACj1vXUjkaSjWTqVL5/oC2IRMBA3MkpQXGhs1t8JfuTIk+Wo8cPThhYouDrRC9jkuoHKkwSdItQNGN7ZGasXhkv5O+o6wGQZ0bPIrCnAnSXXF8A39wo4DLmipxbizNDpgbAwIruOt3lRV00KOGQ8OBukDBZG4VjcewX3SMfRZM4+ROHZgYNICg0Et+mEcRjBl2w/htah+IhpQBG2XIVAInMyA1yvhwNG3HDi56aMx267wyAoxQg/duL84tNIJgUy8V8qz/2uvSVUEzBCUweogo8yGAFK3I6CD4Fu+B5L3xoxl63iJQ3YFee883H8q04BA9fvSgUKP86R/CSRwGT7MsvOtS9PPwP8KrjPtPG+W64wCeKk4ZoXoXheigIMGLKvlQI+08OD7jN2EVYVpJ5WJ7/rp1fbEPGp0iPJIPCy9A+SI/vCaZ9+l1IaXW7/Uw4j7a2lORB3TF9Kg6RYfIHFgvndQzSQGqBhD3JLl2RsS6uUHlMLo8aQoFZhHHfVTB21lGsUk8iBcSlPpsiQP6cQ+SIU2HLkfu4SydexID++Baw7AuwsEaZxc0QaVB8XEWoXQY2dwtb7qGs5RGpbvC5OJ1T3cy/u7UQZLefBcB/2n+ITV+zLyG+EmV+OWnLDAIVw/po8fExmHx9Rjqv3GYm42BlP8fnAvpP76J8E15D0uvTq9P5QOPUvKO3+Z+Ji0GKboKI85V2975Mdn4ASDLON1gZ2SlN3UHLj2CwadOAQefFQ4GaNPA56x3M3x7coSzK+jy1hvJn6geJ9wEh9WB2zenq2N63ARGd6l944/+fTsQlNvbzaIhOJK0rvq+nT7F1BLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAAAAAAAAAAACAAQAAAABjb21tb24vX19pbml0X18ucHlQSwECFAMUAAAACAAAACFcdC3HJdwCAACGCAAADAAAAAAAAAAAAAAAgAEyAAAAY29tbW9uL2lvLnB5UEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAAAAAAAAAAAAAIABOAMAAHBpcGVsaW5lL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXOKzTA5OEwAAI0AAABkAAAAAAAAAAAAAAIABbAMAAHBpcGVsaW5lL3JlZ2VuZXJhdGVfczEucHlQSwECFAMUAAAACAAAACFcYAsn3AMGAADSCwAAIwAAAAAAAAAAAAAAgAHxFgAAY29uZmlncy9yZWdlbmVyYXRlX3MxX2RlZXBzZWVrLnlhbWxQSwUGAAAAAAUABQBUAQAANR0AAAAA'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

## Settings and generation prompt

Defaults come from the bundled `configs/regenerate_s1_deepseek.yaml`, shown below.
Edit `CONFIG` in this cell before loading the source if needed. The default is
**`deepseek-v4-pro`, high thinking effort, 16 concurrent requests** with request pacing
and bounded retries for rate limits and transient API failures. Temperature is omitted
because DeepSeek ignores it in thinking mode. The sampling seed fixes the 20 selected
examples; hosted generation itself is not guaranteed deterministic.

`max_tokens=131072` is an operational ceiling for **reasoning + answer**, not a requested
length or an SFT limit. It is not included in the prompt. A response that hits this ceiling
is flagged as incomplete; its raw text is saved in the journal, but is not placed in the
completed-output columns. No source rows are removed by the old 20,480-token SFT limit.

Changing the config or bundled generation code creates a separate run folder. Reuse the
same notebook/settings to resume. Hosted model aliases may change server-side: returned
model IDs, timestamps, fingerprints, usage, and response IDs are recorded per request.

In [ ]:
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/regenerate_s1_deepseek.yaml").read_text())
# Optional edits go here, before creating the run in the next cell.
# CONFIG["api"]["concurrency"] = ...
print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))

In [ ]:
# @title Load the pinned Hugging Face source and prepare a resumable run
import importlib.metadata
from pipeline.regenerate_s1 import RegenerationRun, load_source, timestamp
from common.io import append_jsonl, read_jsonl
source_rows = load_source(CONFIG, HF_TOKEN)
run = RegenerationRun(source_rows, CONFIG, PROJECT_ROOT)
runtime_path = run.root / "runtime_sessions.jsonl"
read_jsonl(runtime_path, repair_tail=True)
append_jsonl(runtime_path, {"started_at": timestamp(), "python": sys.version,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["datasets", "huggingface-hub", "httpx", "PyYAML", "tqdm"]}})
print("Source rows:", len(source_rows))
print("New columns:", run.reasoning_column, run.answer_column)
print("Smoke sample indices (seeded):", run.smoke_indices)
print("Drive output folder:", run.root)

## Smoke test — 20 random examples and a comparison CSV

This cell makes paid API requests for the 20 sampled rows that are not already complete.
The CSV has exactly these five columns, in this order:

`question`, `deepseek_thinking_trajectory`, `deepseek_attempt`,
`deepseek-v4-pro_reasoning`, `deepseek-v4-pro_answer`.

UTF-8 CSV preserves multiline text, quotation marks, and Unicode. Long reasoning can exceed
spreadsheet apps' cell-display limits; the CSV itself retains the full text. Blank generated
cells mean generation is incomplete/failed/pending or the final-response format is invalid;
inspect `smoke/status.jsonl` for details. We check the four headings in order and nonempty
sections. These structural checks do not assess correctness or quality: review whether the
answer column itself provides the full derivation and meaningful checks. Raw returned text,
including format failures, remains in `attempts.jsonl`. Rerun to retry unsuccessful rows.
The old `deepseek_grade` is not a grade for the new answer, and no grading is run here.

In [ ]:
# @title Smoke test: generate the fixed random sample and save/download the CSV
try:
    await run.generate(run.smoke_indices, DEEPSEEK_API_KEY)
finally:
    smoke_result = run.export(run.smoke_indices, "smoke")
    print(smoke_result["summary"])
    print("CSV saved:", smoke_result["path"])
    print("Row status:", smoke_result["status_path"])
files.download(str(smoke_result["path"]))

## Full run — all 996 rows, preserving original columns

Enable the checkbox below to run. There is no baseline-evaluation or smoke-completion gate.
Matching completed smoke results are reused automatically. After a runtime restart, run
setup/settings/source-loading, then this cell directly to resume.

The export is a Hugging Face-compatible **JSONL dataset** in original source order, with
every original field plus the two generated fields. Existing grades and responses retain
their original meaning; they are never overwritten. The source HF repository is unchanged.

`full/dataset.jsonl` is written when every row is complete. Otherwise the export is called
`full/dataset.partial.jsonl`, still containing every row, with null generated fields for
unsuccessful rows (including missing/empty/out-of-order sections). `full/status.jsonl`
lists failures by source index; `attempts.jsonl`
retains raw returned text, including incomplete responses and retry usage. Rerun the cell
to retry remaining rows. Authentication/balance/invalid-request errors stop the run and
save partial results. Failed responses are not silently promoted into the training data.

This notebook only regenerates data. It does not run SFT, evaluation, or HF publication.

In [ ]:
# @title Full run: generate all rows (completed smoke rows are reused)
RUN_FULL_GENERATION = False # @param {type:"boolean"}
if RUN_FULL_GENERATION:
    full_indices = list(range(len(source_rows)))
    try:
        await run.generate(full_indices, DEEPSEEK_API_KEY)
    finally:
        full_result = run.export(full_indices, "full")
        print(full_result["summary"])
        print("Dataset saved:", full_result["path"])
        print("Row status:", full_result["status_path"])
    if full_result["summary"]["remaining"]:
        print("Some rows remain incomplete. Inspect status.jsonl and rerun this cell.")
    else:
        print("All source rows now have generated reasoning and answers.")
else:
    print("Full run is off. Enable RUN_FULL_GENERATION to process all source rows.")

Load the finished export later with
`load_dataset("json", data_files=".../full/dataset.jsonl", split="train")`.
Drive stores the dataset and journals; save a notebook copy in Drive to retain cell output
displays too. The API journal records returned token usage, including retries. A request
whose response was lost or cancelled can still be billed; consult the provider dashboard.

References: [Kimi k1.5, §2.2](https://arxiv.org/html/2501.12599v1#S2.S2),
[DeepSeek thinking fields](https://api-docs.deepseek.com/guides/thinking_mode/),
[DeepSeek API](https://api-docs.deepseek.com/api/create-chat-completion/).